# 시설 23건과 9개 지역 출처 검토 — 2026-10-02

기록 단위는 출처별 외부 ID와 활성 snapshot이다. 저장된 감사와 행별 판단을 대조한 뒤 현재 DB를 선택적으로 읽기 전용 재감사한다. 외부 자료를 재수집하거나 승인·발행·좌표를 변경하지 않는다. 원문 전체·연락처는 이 노트북에 넣지 않았다.


In [1]:
import json, re, subprocess, tempfile
from pathlib import Path
from collections import Counter
repo=Path.cwd().resolve()
while not (repo/'scripts/tourism/audit.ts').exists():
    if repo.parent==repo: raise RuntimeError('Book해도 저장소 안에서 실행하세요')
    repo=repo.parent
root=repo/'docs/research'
baseline=json.loads((root/'tourism-review-baseline-20261002.json').read_text())
review=json.loads((root/'tourism-facility-review-20261002.json').read_text())
inventory=json.loads((root/'tourism-nine-region-sources-20261002.json').read_text())
checks=json.loads((root/inventory['downloadsFile']).read_text())
key=lambda row:(row['sourceId'],row['externalId'])
unlinked={key(row):row for row in baseline['diagnostics'] if row['placeId'] is None}
assert len(unlinked)==23
assert len(review['rows'])==len({key(row) for row in review['rows']})==len(unlinked)
assert {key(row) for row in review['rows']}==set(unlinked)
for row in review['rows']:
    original=unlinked[key(row)]
    assert all(row[field]==original[field] for field in ['title','snapshotId','contentSha256','latitude','longitude','reason'])
    assert row['approvedPlaceId'] is None and row['officialEvidence']
    if row['decision']=='CONFIRMED_ALIAS_CANDIDATE':
        candidates=[p for p in original['nearby'] if p['id']==row['proposedPlaceId']]
        assert len(candidates)==1 and candidates[0]['distanceMeters']<=250
        assert candidates[0]['website'] is None
assert dict(Counter(row['decision'] for row in review['rows']))==review['summary']
print(json.dumps({'baselineCheckedAt':baseline['checkedAt'],'reviewedFacilities':len(unlinked),'decisions':review['summary'],'appliedAliases':review['appliedAliases']},ensure_ascii=False,indent=2))


{
  "baselineCheckedAt": "2026-10-01T23:31:11.284Z",
  "reviewedFacilities": 23,
  "decisions": {
    "KEEP_SEPARATE": 14,
    "CATALOG_GAP": 6,
    "LOCATION_REVIEW": 1,
    "CONFIRMED_ALIAS_CANDIDATE": 2
  },
  "appliedAliases": 0
}


In [2]:
# Derive the missing app regions from the actual public source registry and app definition.
registry=json.loads((repo/'ops/tourism/sources.json').read_text())
covered={s['regionId'] for s in registry if s['enabled'] and s['rightsStatus']=='approved'}
source=(repo/'server/domain.ts').read_text()
# App region IDs are also guarded against the reviewed nine-region list below.
regions={r['regionId'] for r in inventory['regions']}
expected={'sapporo','otaru','asahikawa-biei','niseko-kutchan','toya-noboribetsu','obihiro-tokachi','kushiro-akan','abashiri-shiretoko','wakkanai-rishiri-rebun'}
assert regions==expected and len(inventory['regions'])==len(regions)==9
assert all(re.search(r"['\"]"+re.escape(region)+r"['\"]",source) for region in regions|covered)
assert covered=={'furano','hakodate','new-chitose'} and not covered & regions
assert inventory['productionEffect']['newEnabledSources']==0
assert inventory['productionEffect']['newPublishedRecords']==0
for region in inventory['regions']:
    candidate=region['primaryCandidate']
    assert candidate['enabled'] is False and candidate['ingestApproved'] is False
print(json.dumps({'regionsWithActiveKnowledge':sorted(covered),'regionsReviewed':len(regions),'newSourcesEnabled':0},ensure_ascii=False,indent=2))


{
  "regionsWithActiveKnowledge": [
    "furano",
    "hakodate",
    "new-chitose"
  ],
  "regionsReviewed": 9,
  "newSourcesEnabled": 0
}


In [3]:
# Download counts are structural evidence, not current opening or visitability guarantees.
files={r['id']:r for r in checks['csvDownloads']}
assert len(files)==3
for record in files.values():
    assert record['status'] in [200,304] and re.fullmatch('[a-f0-9]{64}',record['sha256'])
    assert record['uniqueNonemptyIds']<=record['recordCount']
    assert record['coordinatesValid']<=record['recordCount']
assert files['sapporo']['recordCount']==files['sapporo']['coordinatesValid']==files['sapporo']['uniqueNonemptyIds']==103
assert files['noboribetsu']['emptyIds']==files['noboribetsu']['recordCount']==4
assert files['abashiri']['recordCount']==files['abashiri']['coordinatesValid']==files['abashiri']['uniqueNonemptyIds']==14
asa=checks['asahikawaTourismArchive']
assert sum(asa['categoryCounts'].values())==asa['recordCount']==315
assert asa['coordinateConversionVerified'] is False
assert checks['scenicIndex']['sectionCount']==len(checks['scenicIndex']['sections'])==15
assert checks['scenicIndex']['archiveInspected'] is False
assert len(checks['failedDownloads'])==2 and all(r.get('recordCount') is None and 'error' in r for r in checks['failedDownloads'])
points=checks['scenicPoints']
assert points['archiveInspected'] is True
assert points['kmlFileCount']==len(points['kmlFiles'])==27
assert points['pointOccurrences']==581 and len(points['selectedPoints'])==8
assert len(points['mislabeledZipFiles'])==1
assert points['appliedLinks']==0
for point in points['selectedPoints']:
    assert point['approvedPlaceId'] is None
    for match in point['exactNameMatches']:
        assert match['region_id']==point['reviewedAppRegion'] and match['distance']<=250
assert sum(len(p['exactNameMatches'])==1 for p in points['selectedPoints'])==3
assert {p['reviewedAppRegion'] for p in points['selectedPoints']}=={'otaru','niseko-kutchan','obihiro-tokachi','kushiro-akan','wakkanai-rishiri-rebun'}
print(json.dumps({'files':[{k:r[k] for k in ['id','recordCount','coordinatesValid','emptyIds']} for r in files.values()],'asahikawaRows':asa['recordCount'],'asahikawaCoordinateConversionVerified':False,'scenicIndexSections':15,'scenicPointArchiveInspected':True,'scenicPointOccurrences':581,'selectedPointsProbed':8,'failedHarpCsvDownloads':2},ensure_ascii=False,indent=2))


{
  "files": [
    {
      "id": "sapporo",
      "recordCount": 103,
      "coordinatesValid": 103,
      "emptyIds": 0
    },
    {
      "id": "noboribetsu",
      "recordCount": 4,
      "coordinatesValid": 4,
      "emptyIds": 4
    },
    {
      "id": "abashiri",
      "recordCount": 14,
      "coordinatesValid": 14,
      "emptyIds": 0
    }
  ],
  "asahikawaRows": 315,
  "asahikawaCoordinateConversionVerified": false,
  "scenicIndexSections": 15,
  "scenicPointArchiveInspected": true,
  "scenicPointOccurrences": 581,
  "selectedPointsProbed": 8,
  "failedHarpCsvDownloads": 2
}


## 현재 DB 재감사

다음 셀은 저장소의 Node 의존성 및 PostgreSQL/PostGIS와 `.env` 설정이 필요하다. 설정·비밀값은 출력하지 않는다. 일치 여부는 이번 기준 snapshot에 대한 검증이며 다른 작업에서 DB가 바뀌면 실패할 수 있다. 외부 원문을 다시 요청하지 않는다.


In [4]:
RUN_LIVE_AUDIT=True
if RUN_LIVE_AUDIT:
    with tempfile.TemporaryDirectory() as directory:
        output=Path(directory)/'audit.json'
        subprocess.run(['npm','run','tourism:audit','--','--date','2026-10-02','--output',str(output)],cwd=repo,capture_output=True,text=True,check=True)
        live=json.loads(output.read_text())
    assert live['counts']==baseline['counts']
    assert {(r['sourceId'],r['snapshotId'],r['records'],r['linkedFacilities']) for r in live['profile']}=={(r['sourceId'],r['snapshotId'],r['records'],r['linkedFacilities']) for r in baseline['profile']}
    assert {key(row) for row in live['diagnostics'] if row['placeId'] is None}==set(unlinked)
    print(json.dumps({'checkedAt':live['checkedAt'],'catalogPlaces':live['counts']['catalogPlaces'],'activeFacilities':sum(r['facilities'] for r in live['profile']),'linkedFacilities':sum(r['linkedFacilities'] for r in live['profile']),'unlinkedFacilities':len(unlinked),'databaseUnchangedSinceBaseline':True},ensure_ascii=False,indent=2))
    print(baseline['sql'])
else:
    print('Live audit skipped; saved evidence checks only.')


{
  "checkedAt": "2026-10-02T00:12:11.498Z",
  "catalogPlaces": 20810,
  "activeFacilities": 34,
  "linkedFacilities": 11,
  "unlinkedFacilities": 23,
  "databaseUnchangedSinceBaseline": true
}
SELECT s.id AS "sourceId",v.id AS "snapshotId",
v.content_sha256 AS "contentSha256",v.fetched_at AS "fetchedAt",v.published_at AS "publishedAt",
s.enabled,s.rights_status AS "rightsStatus",v.record_count AS "declaredRecords",
count(r.external_id)::int AS records,
count(*) FILTER (WHERE r.kind='place')::int AS facilities,
count(*) FILTER (WHERE r.kind='event')::int AS events,
count(*) FILTER (WHERE r.kind='place' AND r.latitude IS NOT NULL AND r.longitude IS NOT NULL)::int AS "facilitiesWithCoordinates",
count(*) FILTER (WHERE r.kind='place' AND r.canonical_place_id IS NOT NULL)::int AS "linkedFacilities",
count(*) FILTER (WHERE r.kind='event' AND r.latitude IS NOT NULL AND r.longitude IS NOT NULL)::int AS "eventsWithCoordinates",
count(*) FILTER (WHERE r.kind='event' AND r.date_status IN ('confi

## 해석 범위

별칭 후보 2건의 판단은 실제 승인 파일·DB 적용과 다르다. 현재 미연결은 23건이다. 9개 지역 모두의 출처 후보를 확인했다. 시정촌 4개 자료 파일과 5개 지역의 선택 지점을 포함하는 공통 지도 파일을 읽었다. HARP CSV 두 개는 시간 초과됐고, 명칭·좌표·ID 미리보기 확인과 파일 확보는 구분한다. 서비스 source 등록·정제·발행은 하지 않았다. 전화·메일·사진·요금·현재 운영 표현을 가져오지 않는다.
